# Task 5 — Source Metadata Ingestion into MongoDB

**Points**: 2.0 / 10  
**Script**: [`src/spark_streaming_job.py`](../../src/spark_streaming_job.py)

---

## 5.1 Design Rationale

Unlike Task 4 (Neo4j, ingested directly via the Kafka Connector Sink with **no** Spark layer), the lab spec explicitly requires Task 5 to go through **Apache Spark Structured Streaming**. We built a single streaming query that:

1. Subscribes to the `cpg.metadata` Kafka topic (1 partition, 1 message per source file)
2. Parses each JSON message against a fixed `StructType` schema
3. Writes the parsed rows into MongoDB (`cpg_db.cpg_metadata`) through the **MongoDB Spark Connector**
4. Uses a **checkpoint location** on disk (`checkpoints/cpg-metadata`) so the job can resume from the last processed Kafka offset after a restart

```
cpg.metadata (Kafka, 1 partition) ──► Spark Structured Streaming ──► MongoDB (cpg_db.cpg_metadata)
                                          │
                                   checkpoints/cpg-metadata
                                   (offsets + commit log on disk)
```

### Idempotency design

The MongoDB Spark Connector **upserts by `_id`** by default. We rename the incoming `file_path` field to `_id` before writing:

```python
out = parsed.withColumnRenamed("file_path", "_id")
```

Since `file_path` is unique per source file, replaying the same file's metadata event always overwrites the same document instead of inserting a new one — this is what Task 6 later verifies experimentally.

## 5.2 Job Configuration

Key Spark session / writer settings from `spark_streaming_job.py`:

```python
PACKAGES = ",".join([
    "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.1",
    "org.mongodb.spark:mongo-spark-connector_2.12:10.3.0",
])

raw = (
    spark.readStream.format("kafka")
    .option("kafka.bootstrap.servers", "localhost:29092")
    .option("subscribe", "cpg.metadata")
    .option("startingOffsets", "earliest")
    .load()
)

writer = (
    out.writeStream.format("mongodb")
    .option("spark.mongodb.connection.uri", MONGO_URI)
    .option("spark.mongodb.database", "cpg_db")
    .option("spark.mongodb.collection", "cpg_metadata")
    .option("checkpointLocation", str(CHECKPOINT_DIR))
    .outputMode("append")
)
```

The job supports two modes: `trigger(availableNow=True)` (process everything currently in the topic, then exit — used below for verification) and `--continuous` (`trigger(processingTime="10 seconds")`, a long-lived query for a real deployment).

## 5.3 Producing the Metadata Events

Before the Spark job has anything to read, the Parser Service (Task 2) must have emitted the per-file metadata events into `cpg.metadata`. Re-running it against the 545 included files of `huggingface/lerobot`:

In [1]:
# python src/parser_service.py
# (545 included files vs. the 543 from the team's original snapshot — the
#  upstream huggingface/lerobot repo gained a couple of files between the
#  two clones; this does not affect the pipeline design.)

# Captured from: python src/parser_service.py  (run on 2026-07-22)
[50/545] processed (3.7s elapsed)
[100/545] processed (7.9s elapsed)
[150/545] processed (15.4s elapsed)
[200/545] processed (20.8s elapsed)
[250/545] processed (34.1s elapsed)
[300/545] processed (40.0s elapsed)
[350/545] processed (44.6s elapsed)
[400/545] processed (47.8s elapsed)
[450/545] processed (52.9s elapsed)
[500/545] processed (55.4s elapsed)
[545/545] processed (58.3s elapsed)

Done. 545 files parsed OK, 0 errors, 441635 nodes, 191429 edges emitted.


A sample raw message from `cpg.metadata` (consumed directly from the broker to confirm the producer side is correct before Spark even runs):

# docker exec -it cpg-kafka kafka-console-consumer \
#   --bootstrap-server localhost:29092 --topic cpg.metadata --from-beginning --max-messages 1

{"schema_version": "1.0", "event_time": "2026-07-22T08:59:02.474030+00:00", "event_type": "metadata", "file_path": "lerobot/examples/annotations/run_hf_job.py", "file_hash": "f5aad3e886a796e3e885b36ba8f511ebdd1540058cdf61073e0526e4a0b3ccb0", "size_bytes": 3158, "loc": 81, "num_functions": 0, "num_classes": 0, "num_nodes": 62, "num_edges": 21, "_key": "lerobot/examples/annotations/run_hf_job.py"}


## 5.4 Running the Spark Structured Streaming Job

With `SPARK_HOME` unset (so PySpark uses the JAR set bundled with the `pyspark==3.5.1` pip package rather than a locally installed Spark 4.x, which caused a `SparkSession` constructor mismatch on first attempt), the job is run in `availableNow` mode to process everything currently sitting in `cpg.metadata`:

# python src/spark_streaming_job.py   (batch 0, first run)

Streaming query finished. Final progress: {
  'id': 'dabaa5ba-2403-4d20-8316-bf1471cc3ea8',
  'runId': '0c416230-4649-4174-8234-17311a9b612a',
  'batchId': 0,
  'numInputRows': 545,
  'sources': [{
      'description': 'KafkaV2[Subscribe[cpg.metadata]]',
      'startOffset': None,
      'endOffset': {'cpg.metadata': {'0': 545}},
      'numInputRows': 545
  }],
  'sink': {
      'description': 'MongoTable{... spark.mongodb.collection=cpg_metadata, spark.mongodb.database=cpg_db, checkpointLocation=.../checkpoints/cpg-metadata ...}',
      'numOutputRows': 545
  }
}


`numInputRows: 545` and `numOutputRows: 545` confirm every metadata event from the topic was read and written — no messages were dropped, and none were duplicated at the source (a single Kafka partition guarantees strict per-file ordering, matching the `file_path` key used at production time in Task 3).

## 5.5 Verifying the Result in MongoDB

In [4]:
from pymongo import MongoClient

client = MongoClient("mongodb://localhost:27017")
db = client.cpg_db
print("Document count:", db.cpg_metadata.count_documents({}))
print(db.cpg_metadata.find_one({"_id": "lerobot/examples/annotations/run_hf_job.py"}))

Document count: 545
{'_id': 'lerobot/examples/annotations/run_hf_job.py', 'schema_version': '1.0', 'event_time': '2026-07-22T08:59:02.474030+00:00', 'event_type': 'metadata', 'file_hash': 'f5aad3e886a796e3e885b36ba8f511ebdd1540058cdf61073e0526e4a0b3ccb0', 'size_bytes': 3158, 'loc': 81, 'num_functions': 0, 'num_classes': 0, 'num_nodes': 62, 'num_edges': 21}


545 documents in `cpg_db.cpg_metadata`, one per included source file, with `_id` set to `file_path` and all metadata fields (`file_hash`, `loc`, `num_functions`, `num_classes`, `num_nodes`, `num_edges`) intact end-to-end from the Parser Service through Kafka and Spark into MongoDB.

*(Insert a MongoDB Compass / `mongosh` screenshot of the `cpg_metadata` collection here for the final submission.)*

## 5.6 Reflection

### What worked
- `foreachBatch`-free `writeStream.format("mongodb")` with `outputMode("append")` worked directly once the `_id`-rename trick was in place — no custom upsert logic needed, the connector's default `_id` matching handled it.
- `trigger(availableNow=True)` made verification deterministic: the job processes exactly what's in the topic and exits, rather than running indefinitely, which is easier to reason about for a lab report.

### Issues encountered
- First run failed with `py4j.Py4JException: Constructor org.apache.spark.sql.SparkSession(...) does not exist`. Root cause: a `SPARK_HOME` environment variable pointed at a locally-installed **Spark 4.1.1**, which conflicted with the **PySpark 3.5.1** Python API installed via `pip`. `unset SPARK_HOME` fixed it — PySpark then used the JAR set bundled inside its own pip package instead.
- Because the Kafka container has no persistent volume in `docker-compose.yml`, every `docker compose down`/`up` cycle wipes all topic data, so the Parser Service (Task 2) had to be re-run to regenerate `cpg.metadata` before Task 5 had anything to consume.

### Lessons learned
- Environment variables like `SPARK_HOME` silently override the Python-side Spark package and produce confusing JVM-level errors rather than a clear "version mismatch" message — worth checking `echo $SPARK_HOME` early when PySpark throws `Py4JException`.
- Keying the Mongo document `_id` on a natural key (`file_path`) rather than an auto-generated ObjectId is what makes the whole pipeline idempotent — this single design decision is what Task 6 depends on.